# 动手实践：在 AMD Radeon™ GPU 上用 RLinf 对视觉-语言-动作（VLA）机器人策略做强化学习

在这个 workshop 里，你将在物理仿真器中用**强化学习（RL）**对一个**视觉-语言-动作（VLA）**机器人策略做端到端的微调，硬件是 **4 张 AMD Radeon™ PRO W7900D GPU**。

| 组件 | 本次使用 |
| --- | --- |
| RL 框架 | [RLinf](https://github.com/RLinf/RLinf)，面向具身智能和智能体的可扩展 RL 基础设施 |
| 策略 | **Lingbot-VLA**（42 亿参数），已在 RoboTwin 任务上做过监督微调（SFT） |
| 仿真器 | [RoboTwin 2.0](https://github.com/RoboTwin-Platform/RoboTwin)（SAPIEN 物理仿真 + Vulkan 渲染），双臂 ALOHA 机器人 |
| 任务 | `place_shoe`：拿起一只鞋，放到目标垫子上 |
| 算法 | **GRPO**（Group Relative Policy Optimization，组相对策略优化），奖励为任务是否成功 |
| 硬件 / 软件 | 4 × Radeon PRO W7900D（gfx1100，48 GB），ROCm 7.2，PyTorch 2.11 |

**接下来约 40 分钟你会做的事**

1. 检查容器里的 GPU 和软件环境（约 3 分钟）。
2. 了解关键的训练参数（约 5 分钟）。
3. 启动一个 **40 步的 GRPO 训练**，**每 10 步**在未参与训练的场景上评估一次。第一次评估结果在启动后**约 18 分钟**出来（每个训练步约 55–75 秒），之后约每 12 分钟出一次新结果，workshop 期间能看到两到三次评估。整个训练约 55 分钟。
4. 实时查看训练进度、理解各项指标，并播放每次评估的视频。

> 这是一个*演示规模*的训练：我们减少了并行环境的数量，让每 10 步（含一次评估）能在 20 分钟内完成。

## 背景：用 RL 微调 VLA 策略

VLA 策略把**相机图像 + 机器人状态 + 一条语言指令**映射为机器人的**动作**。Lingbot-VLA 每次预测一个*动作块（action chunk）*：两只机械臂接下来 50 步的关节空间动作。仿真器执行这个动作块、渲染新的相机图像，策略再预测下一个动作块。`place_shoe` 的一个回合（episode）最多 400 个动作，也就是 8 个动作块。

监督微调（SFT）让策略模仿示范数据。**RL 微调**则让策略在仿真器里反复练习，强化那些真正完成了任务的行为。

### 用于机器人策略的 GRPO

对每个初始场景（物体位置、光照、纹理），RLinf 用当前策略跑一**组**共 $G$ 个回合，并给每个回合一个奖励 $R_i$，这里就是任务是否成功：

$$R_i = \mathbb{1}[\text{回合 } i \text{ 成功}], \qquad i = 1, \dots, G$$

GRPO 不训练价值网络（critic），而是用同一组的结果作为基线：

$$\hat A_i = \frac{R_i - \operatorname{mean}(R_1, \dots, R_G)}{\operatorname{std}(R_1, \dots, R_G)}$$

同组里别人失败而自己成功的回合会得到正的优势（advantage）。然后对已执行动作的对数概率，用 PPO 的裁剪策略梯度目标更新策略：

$$\mathcal{L}(\theta) = -\,\mathbb{E}\Big[\min\big(\rho_t \hat A,\ \operatorname{clip}(\rho_t, 1-\epsilon_\text{low}, 1+\epsilon_\text{high})\,\hat A\big)\Big], \qquad \rho_t = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_\text{old}}(a_t \mid s_t)}$$

这个配方使用非对称的裁剪范围：$\epsilon_\text{low} = 0.2$，$\epsilon_\text{high} = 0.28$（`algorithm.clip_ratio_low` / `clip_ratio_high`）。

如果一组里所有回合都成功（或都失败），所有优势都是 0，这一组就学不到东西。这也是为什么一个**很强的 SFT 策略**（成功率约 90%）仍然能从 RL 中获益：剩下的那些失败正是 GRPO 学习的来源。

本次 workshop 只训练 Lingbot-VLA 的 **action expert**（42 亿参数中的约 4.4 亿），视觉-语言主干保持冻结。

### RLinf 如何执行一个训练步

RLinf 用 Ray 启动三组 worker，全部放在同样的 4 张 GPU 上（每张 GPU 上每种 worker 各一个）：

```text
  每张 GPU 上（共 4 张）各有一个：

                 ┌──────────────────────────────────────────────────────────────────────────┐
                 │                                                                          │
  RoboTwin <---> │  EnvWorker                MultiStepRolloutWorker      EmbodiedFSDPActor  │
                 │                                                                          │
                 └──────────────────────────────────────────────────────────────────────────┘

  EnvWorker               运行 RoboTwin 场景：SAPIEN 物理仿真（CPU），Vulkan 相机渲染（GPU）
  MultiStepRolloutWorker  策略推理：生成动作块和对数概率
  EmbodiedFSDPActor       用 FSDP 训练策略

  每个训练步：
    1. rollout    EnvWorker ⇄ RolloutWorker：每个环境跑一个回合，共 8 轮
                  “观测 → 预测 50 个动作 → 仿真执行”
    2. advantage  根据成功奖励计算组相对优势
    3. update     Actor：用收集到的轨迹做策略梯度更新
    4. sync       Actor → RolloutWorker：把更新后的权重发给下一轮 rollout
  每 10 步，在更新之后：
    5. eval       用当前策略在 32 个未参与训练的场景上评估（不训练）
```

某个阶段里空闲的 worker 会让出 GPU：actor 和 rollout worker 把各自的那份策略模型移到 CPU 内存（**offload，卸载**），env worker 则关闭仿真环境、在下一轮 rollout 时重新创建，这样三者可以共用每张 48 GB 的 GPU。

## 1. 检查 AMD GPU 和软件环境

### 检查 GPU

`amd-smi` 是 AMD 用来监控和管理 GPU 的命令行工具，相当于 NVIDIA 的 `nvidia-smi`。不带参数运行时，它会列出容器里可见的每张 GPU：显存用量/总量（`Mem-Usage`）、利用率（`GFX-Uti`）、温度、功耗，以及正在使用 GPU 的进程。`amd-smi monitor` 则把每张 GPU 显示成紧凑的一行。

In [ ]:
!amd-smi

### GPU 和路径

下一个 cell 设置本次训练使用的 GPU 和路径，并检查环境是否就绪，不需要改动。

- **GPU**：这台实例的 4 张 GPU（`GPU_GROUP = "0-3"`）。
- **路径**：镜像里的代码和 Python 环境（`/opt/...`），只读数据卷上的模型和场景资源（`/models/rlinf-workshop`），以及本次训练的输出目录（`/workspace/rlinf_workshop`）。

正常输出：每一行都是“找到”，4 张 GPU 的占用接近 0 %，最后一行是“所有路径都已找到。”。如果出现“缺失”或“<-- 已被占用？”，请找讲师。

In [ ]:
import os
import subprocess
from pathlib import Path

# ---- 训练使用的 4 张 GPU ----
GPU_GROUP = "0-3"

# ---- workshop 容器里的路径（由组织者准备）----
RLINF_ROOT    = "/opt/RLinf"                           # RLinf 代码
VENV          = "/opt/venv/lingbotvla"                 # Lingbot-VLA 的 Python 环境
ROBOTWIN_PATH = "/opt/robotwin"                        # RoboTwin 代码；资源在 $ROBOTWIN_PATH/assets

def find_model(name):
    # 模型在只读数据卷 /models/rlinf-workshop 上（放在 /models 或 /data/models 下也能找到）
    for pattern in (f"/models/rlinf-workshop/{name}", f"/models/{name}", f"/models/*/{name}",
                    f"/data/models/{name}", f"/data/models/*/{name}"):
        found = sorted(Path("/").glob(pattern.lstrip("/")))
        if found:
            return str(found[0])
    return f"/models/rlinf-workshop/{name}"

POLICY_CKPT   = find_model("lingbot-vla-4b-posttrain-robotwin")   # RoboTwin 的 SFT 策略（权重）
TOKENIZER     = find_model("Qwen2.5-VL-3B-Instruct")              # tokenizer / processor
LINGBOT_CFG   = find_model("lingbot-vla-4b")                      # Lingbot-VLA 模型配置
WORK_DIR      = "/workspace/rlinf_workshop"            # 训练的日志、指标和视频

RUN_NAME = f"lingbot_place_shoe_grpo_gpu{GPU_GROUP.replace('-', '_')}"
RUN_DIR  = f"{WORK_DIR}/{RUN_NAME}"

# 导出为环境变量，供下面的 %%bash cell 使用
for k, v in dict(GPU_GROUP=GPU_GROUP, RLINF_ROOT=RLINF_ROOT, VENV=VENV, ROBOTWIN_PATH=ROBOTWIN_PATH,
                 POLICY_CKPT=POLICY_CKPT, TOKENIZER=TOKENIZER, LINGBOT_CFG=LINGBOT_CFG,
                 WORK_DIR=WORK_DIR, RUN_NAME=RUN_NAME, RUN_DIR=RUN_DIR).items():
    os.environ[k] = v

ok = True
for name, path in [("RLinf", RLINF_ROOT), ("Python 环境", VENV), ("RoboTwin", ROBOTWIN_PATH),
                   ("RoboTwin 资源", f"{ROBOTWIN_PATH}/assets/objects"), ("策略 checkpoint", POLICY_CKPT),
                   ("tokenizer", TOKENIZER), ("Lingbot-VLA 配置", LINGBOT_CFG)]:
    exists = Path(path).exists()
    ok &= exists
    print(f"{'找到' if exists else '缺失'}  {path:58s}  {name}")

first, last = (int(x) for x in GPU_GROUP.split("-"))
gpus = list(range(first, last + 1))
print(f"\nGPU 组 {GPU_GROUP} -> GPU {gpus}")
csv = subprocess.run(["rocm-smi", "--showuse", "--showmemuse", "--csv"], capture_output=True, text=True).stdout
for line in csv.splitlines():
    if line.startswith("card") and int(line.split(",")[0][4:]) in gpus:
        card, use, vram = line.split(",")[:3]
        print(f"  {card}: GPU use {use:>3}%  VRAM {vram:>3}%" + ("   <-- 已被占用？" if int(vram) > 5 else ""))
print("\n所有路径都已找到。" if ok else "\n有路径缺失：请先找讲师确认再继续。")

### 检查 Python 环境

容器里每个模型在 `/opt/venv` 下有一个独立的 Python 环境。激活 Lingbot-VLA 的环境，检查 PyTorch 能否通过 ROCm/HIP 看到 AMD GPU，以及 RLinf 和仿真器能否正常导入：

In [ ]:
%%bash
set -e
source "${VENV}/bin/activate"
cd "${RLINF_ROOT}"
export PYTHONPATH="${RLINF_ROOT}:${ROBOTWIN_PATH}"

(python - <<'PY'
import torch
print("PyTorch      :", torch.__version__, "| HIP", torch.version.hip)
print("GPUs visible :", torch.cuda.device_count(), "x", torch.cuda.get_device_name(0))
import sapien, rlinf, os
print("SAPIEN       :", sapien.__version__)
print("RLinf        :", os.path.dirname(rlinf.__file__))
from rlinf.envs.sim.robotwin.robotwin_env import RoboTwinEnv  # noqa: F401
print("RoboTwin env : import OK")
print("flash-attn backend:", "Triton" if os.environ.get("FLASH_ATTENTION_TRITON_AMD_ENABLE") == "TRUE" else "CK")
PY
) 2>&1 | grep -v "(null): No such file or directory"

## 2. 训练配置

RLinf 的配置是 Hydra YAML 文件。我们以配方
`examples/embodiment/config/robotwin_place_shoe_grpo_lingbotvla.yaml` 为基础，在命令行上覆盖少数几个值。先打印出配方里和本次相关的部分：

In [ ]:
%%bash
source "${VENV}/bin/activate"
python - <<'PY'
import os
from omegaconf import OmegaConf
cfg = OmegaConf.load(os.path.join(os.environ["RLINF_ROOT"], "examples/embodiment/config/robotwin_place_shoe_grpo_lingbotvla.yaml"))
show = {
    "cluster.component_placement": cfg.cluster.component_placement,
    "runner": {k: cfg.runner[k] for k in ("max_steps", "val_check_interval", "save_interval")},
    "algorithm": {k: cfg.algorithm[k] for k in ("adv_type", "group_size", "update_epoch", "clip_ratio_low", "clip_ratio_high")},
    "env.train": {k: cfg.env.train[k] for k in ("total_num_envs", "rollout_epoch", "max_episode_steps", "group_size")},
    "env.eval": {k: cfg.env.eval[k] for k in ("total_num_envs", "rollout_epoch", "max_episode_steps")},
    "rollout": {k: cfg.rollout[k] for k in ("backend", "pipeline_stage_num", "enable_offload")},
    "actor": {"micro_batch_size": cfg.actor.micro_batch_size, "global_batch_size": cfg.actor.global_batch_size,
              "optim.lr": cfg.actor.optim.lr, "enable_offload": cfg.actor.enable_offload},
}
print(OmegaConf.to_yaml(OmegaConf.create({k.replace(".", "/"): v for k, v in show.items()})))
PY

### workshop 覆盖了哪些参数，为什么

完整配方在 128 个并行环境上用大批量训练，一步要好几分钟。workshop 保持算法不变，缩小批量，让每 10 步（含一次评估）在 20 分钟内完成。

| 覆盖项 | 配方默认值 | workshop | 原因 |
| --- | --- | --- | --- |
| `cluster.component_placement` | `actor, env, rollout: 0-7` | `{actor: <组>, env: <组>, rollout: <组>}` | 只用你的 4 张 GPU；每张 GPU 上每种 worker 各一个 |
| `env.train.total_num_envs` | 128 | **16** | 每张 GPU 4 个环境：每步耗时随每张 GPU 上的环境数增加 |
| `algorithm.group_size` | 8 | **4** | GRPO 每组的回合数；16 个环境即每步 4 组 |
| `env.train.rollout_epoch` | 8 | **1** | 每步每个环境跑一个回合 |
| `actor.global_batch_size` / `actor.micro_batch_size` | 1024 / 32 | **16 / 4** | 16 条轨迹组成一个全局批次；`global_batch_size` 必须能被 `micro_batch_size × 4 张 GPU` 整除 |
| `algorithm.update_epoch` | 2 | **1** | 每个批次只训练一遍 |
| `env.eval.total_num_envs` | 128 | **32** | 每次评估 32 个未参与训练的回合 |
| `runner.max_steps` / `runner.val_check_interval` | -1 / 20 | **40 / 10** | 训练 40 步，在第 10、20、30、40 步后评估 |
| `runner.save_interval` | 20 | **-1** | 不保存 checkpoint（每个约 18 GB） |
| `env.eval.video_cfg.save_video` / `fps` | True / 30 | **True / 2** | 保存评估视频；每个动作块录一帧，以 2 fps 播放 |
| `env.train.video_cfg.save_video` | True | **False** | 不保存训练视频 |

RLinf 启动时会检查两个约束：

- `total_num_envs / GPU 数`必须能被 `group_size` 整除（每张 GPU 上放的是完整的组），
- `global_batch_size` 必须能被 `micro_batch_size × GPU 数`整除。

> **在 AMD Radeon GPU 上，请保持 `rollout.pipeline_stage_num: 1`（默认值），并且不要开启 `runner.overlap_env_bootstrap`。** 这两个选项都会让 Vulkan 渲染和 HIP 计算同时在同一张 GPU 上运行，在 Radeon PRO W7900D 上曾导致 GPU 挂起（`HW Exception ... GPU Hang`、`vk::Device::waitForFences: ErrorDeviceLost`）。而且它们在这里节省的时间很少：每步的耗时主要花在重置 RoboTwin 场景上。

## 3. 启动训练

下一个 cell 会在**后台**启动训练并立即返回。它运行 RLinf 的具身智能训练脚本，使用第 2 节的配方和上表中 workshop 覆盖的参数：

```bash
cd /opt/RLinf
python examples/embodiment/train_embodied_agent.py \
    --config-path /opt/RLinf/examples/embodiment/config \
    --config-name robotwin_place_shoe_grpo_lingbotvla \
    <workshop 覆盖的参数>
```

`train_embodied_agent.py` 会启动 Ray，在你的 4 张 GPU 上启动 env、rollout 和 actor 三组 worker，然后执行“RLinf 如何执行一个训练步”中介绍的训练循环。在此之前，cell 会先激活 Python 环境，并把 RLinf 和 RoboTwin 加入 `PYTHONPATH`。

日志写到 `$RUN_DIR/train.log`，指标写到 `$RUN_DIR/metrics.log`。训练正在运行时重复执行这个 cell 不会再启动一份。

In [ ]:
%%bash
set -euo pipefail
mkdir -p "${RUN_DIR}"
LOG_FILE="${RUN_DIR}/train.log"
PID_FILE="${RUN_DIR}/train.pid"

# 进程不存在或是僵尸进程（已结束但尚未被回收）时，视为没有在运行
is_running() { local s; s=$(ps -o stat= -p "$1" 2>/dev/null); [[ -n "${s}" && "${s}" != Z* ]]; }
if [[ -f "${PID_FILE}" ]] && is_running "$(cat "${PID_FILE}")"; then
    echo "训练已经在运行（PID $(cat "${PID_FILE}")）。日志：${LOG_FILE}"
    exit 0
fi

nohup setsid bash -s >"${LOG_FILE}" 2>&1 <<'TRAIN_SCRIPT' &
set -uo pipefail
source "${VENV}/bin/activate"
cd "${RLINF_ROOT}"
export PYTHONPATH="${RLINF_ROOT}:${ROBOTWIN_PATH}"
export REPO_PATH="${RLINF_ROOT}" EMBODIED_PATH="${RLINF_ROOT}/examples/embodiment"
export ROBOT_PLATFORM=ALOHA ROBOTWIN_PATH="${ROBOTWIN_PATH}"
export LINGBOT_VLA_PATH="${LINGBOT_VLA_PATH:-${VENV}/lingbot-vla}"
export MUJOCO_GL=egl PYOPENGL_PLATFORM=egl TOKENIZERS_PARALLELISM=false PYTHONUNBUFFERED=1 HYDRA_FULL_ERROR=1

echo "TRAIN_START $(date -u '+%F %T') UTC  GPU group ${GPU_GROUP}"
python examples/embodiment/train_embodied_agent.py \
    --config-path "${RLINF_ROOT}/examples/embodiment/config" \
    --config-name robotwin_place_shoe_grpo_lingbotvla \
    '~cluster.component_placement' \
    "+cluster.component_placement={actor:${GPU_GROUP},env:${GPU_GROUP},rollout:${GPU_GROUP}}" \
    runner.logger.log_path="${RUN_DIR}" \
    runner.max_steps=40 runner.val_check_interval=10 runner.save_interval=-1 \
    algorithm.group_size=4 algorithm.update_epoch=1 \
    env.train.total_num_envs=16 env.train.rollout_epoch=1 \
    env.eval.total_num_envs=32 \
    actor.global_batch_size=16 actor.micro_batch_size=4 \
    env.train.assets_path="${ROBOTWIN_PATH}" env.eval.assets_path="${ROBOTWIN_PATH}" \
    env.train.video_cfg.save_video=False \
    env.eval.video_cfg.save_video=True +env.eval.video_cfg.fps=2 \
    actor.model.model_path="${POLICY_CKPT}" \
    actor.model.tokenizer_path="${TOKENIZER}" \
    actor.model.lingbotvla.config_path="${LINGBOT_CFG}"
echo "TRAIN_EXIT=$?  $(date -u '+%F %T') UTC"
TRAIN_SCRIPT

echo $! > "${PID_FILE}"
echo "训练已在后台启动。"
echo "  PID ：$(cat "${PID_FILE}")"
echo "  日志：${LOG_FILE}"

### 实时查看训练

运行下一个 cell 查看训练进度。它每 15 秒刷新一次，显示：

- **状态**：运行中还是已完成、距启动的时间、已完成的步数（共 40 步）。
- **阶段**：训练当前在做什么：启动中、rollout、策略更新或评估。
- **评估成功率**：第 10、20、30、40 步后的评估结果。
- **RLinf 的指标表**：来自 `$RUN_DIR/metrics.log`，每完成一步一张，最新的在最下面（第 4 节解释各项指标）。

**一有新的评估结果，这个 cell 会自动停止**。这时可以到第 5 节看新一次评估的视频，然后再运行这个 cell 继续查看。你也可以随时点 **Stop（■）** 停止它：这只是停止查看，**训练仍在继续**。

4 × W7900D 上的预计时间：

| 阶段 | 时间 |
| --- | --- |
| 启动：启动 Ray、加载模型、创建环境 | 约 2.5 分钟 |
| 第 1 步（含第一次推理和权重同步） | 约 130 秒 |
| 之后每一步 | 约 55–75 秒 |
| 每次评估（32 个回合） | 第一次约 3 分钟，之后约 1.5 分钟；如果其他任务占用 CPU 会更久 |
| **第一次评估结果（第 10 步后）** | **启动后约 18 分钟** |
| 之后的评估 | 约每 12 分钟一次 |
| 40 步全部完成 | 约 55 分钟 |

In [ ]:
import calendar, re, time
from IPython.display import clear_output, display

ANSI = re.compile(r"\x1b\[[0-9;]*m")

def parse_metrics(path):
    # 把 RLinf 打印的指标表解析成每步一个字典
    try:
        txt = ANSI.sub("", Path(path).read_text(errors="ignore"))
    except FileNotFoundError:
        return []
    rows = []
    for block in txt.split("Metric Table")[1:]:
        m = re.search(r"Global Step:\s+(\d+)/(\d+)", block)
        if not m or "╰" not in block:  # 还没有底边框：RLinf 还在写这张表
            continue
        row, section = {"step": int(m.group(1)), "of": int(m.group(2))}, ""
        for line in block.splitlines():
            h = re.search(r"├─+\s*([A-Za-z/ ]+?)\s*─+┤", line)
            if h:
                section = h.group(1).strip()
                continue
            for k, v in re.findall(r"([A-Za-z_/]+)=([-0-9.e+]+)", line):
                try:
                    row[f"{section}:{k}"] = float(v)
                except ValueError:
                    pass
        rows.append(row)
    return rows

def training_status():
    pid_file, log_file = Path(RUN_DIR, "train.pid"), Path(RUN_DIR, "train.log")
    log = ANSI.sub("", log_file.read_text(errors="ignore")) if log_file.exists() else ""
    m = re.search(r"TRAIN_EXIT=(\d+)", log)
    if m:
        return ("已完成" if m.group(1) == "0" else f"失败（退出码 {m.group(1)}）"), log
    stat = subprocess.run(["ps", "-o", "stat=", "-p", pid_file.read_text().strip()],
                          capture_output=True, text=True).stdout.strip() if pid_file.exists() else ""
    alive = bool(stat) and not stat.startswith("Z")  # 僵尸进程表示已结束、但还没被回收
    return ("运行中" if alive else "未运行"), log

def current_phase(log, rows):
    # 每一步结束时日志里会打印一张指标表：只看最后一张表之后的内容
    current = log.split("Metric Table")[-1].splitlines()
    evals = [line for line in current if "Evaluating Rollout Epochs" in line]
    rollouts = [line for line in current if "Generating Rollout Epochs" in line]
    if evals:
        return "评估中（32 个 episode）" if "100%" not in evals[-1] else "评估完成，正在结束这一步"
    if rollouts:
        return "rollout 中（16 个环境）" if "100%" not in rollouts[-1] else "策略更新中（训练 action expert）"
    return "下一步即将开始" if rows else "启动中：启动 Ray、加载模型、创建环境"

def since_launch(log):
    m = re.search(r"TRAIN_START (\S+ \S+) UTC", log)
    if not m:
        return "-"
    start = calendar.timegm(time.strptime(m.group(1), "%Y-%m-%d %H:%M:%S"))  # 日志里的时间是 UTC
    return time.strftime("%M:%S" if time.time() - start < 3600 else "%H:%M:%S", time.gmtime(time.time() - start))

def show(rows, status, log):
    clear_output(wait=True)
    done = rows[-1]["step"] if rows else 0
    total = rows[-1]["of"] if rows else 40
    header = f"[{time.strftime('%H:%M:%S')}] 状态：{status} | 距启动 {since_launch(log)} | 已完成 {done}/{total} 步"
    print(header)
    print(f"阶段：{current_phase(log, rows)}")
    evals = [(r["step"], r["Evaluation:success_once"]) for r in rows if "Evaluation:success_once" in r]
    if evals:
        print("评估成功率：" + "   ".join(f"第 {s} 步 {v:.3f}" for s, v in evals))
    metrics_log = Path(RUN_DIR, "metrics.log")
    if not metrics_log.exists():
        print("\n还没有指标表：RLinf 每完成一步会往 metrics.log 写一张。")
        return
    print(f"\n{metrics_log}（RLinf 每完成一步写入的指标表，按时间顺序）：")
    print(metrics_log.read_text(errors="ignore"))
    print(header)  # 在最新一张表下面再显示一次

def eval_steps(rows):
    return [r["step"] for r in rows if "Evaluation:success_once" in r]

seen = eval_steps(parse_metrics(Path(RUN_DIR, "metrics.log")))  # 本次运行这个 cell 之前已有的评估
try:
    while True:
        status, log = training_status()
        rows = parse_metrics(Path(RUN_DIR, "metrics.log"))
        show(rows, status, log)
        new = [s for s in eval_steps(rows) if s not in seen]
        if new:
            print(f"\n第 {new[-1]} 步后的评估结果已出来：到第 5 节看这次评估的视频，"
                  "然后再运行这个 cell 继续查看。")
            break
        if status != "运行中":
            break
        time.sleep(15)
except KeyboardInterrupt:
    print("\n已停止查看（训练仍在后台运行）。")

**两个日志文件。** 训练会往 `$RUN_DIR`（`/workspace/rlinf_workshop/lingbot_place_shoe_grpo_gpu0_3`）写两个文件：

- `train.log`：训练打印的全部内容：启动信息、进度条、各个 worker 的警告和错误，以及指标表。出问题时看这个文件。
- `metrics.log`：只有 RLinf 写入的每一步的指标表。上面的 cell 显示的就是这个文件。

想实时查看 `train.log`，可以打开一个终端（**File → New → Terminal**），运行：

```bash
tail -f /workspace/rlinf_workshop/lingbot_place_shoe_grpo_gpu0_3/train.log
```

按 **Ctrl+C** 停止查看，训练仍会继续。

## 4. 理解训练指标

RLinf 每一步结束后都会打印一张指标表（同时写到 `metrics.log`，并以 TensorBoard 事件保存在 `$RUN_DIR/tensorboard`）。最有用的几项：

**环境与评估（任务表现）**

| 指标 | 含义 |
| --- | --- |
| `Environment:success_once` | 本步训练回合中，至少成功过一次的比例 |
| `Environment:num_trajectories` | 本步收集的训练回合数（16） |
| `Evaluation:success_once` | 在未参与训练的评估场景上的成功率，**最关键的数字** |
| `Evaluation:success_at_end` | 评估回合在最后一步时仍处于成功状态的比例 |
| `Evaluation:episode_len` | 回合长度，按动作数计（400 即上限；这里总是 400，因为每个回合都会跑到上限） |

**Rollout 与训练（RL 信号）**

| 指标 | 含义 |
| --- | --- |
| `Rollout:advantages_mean/min/max` | 组相对优势；全为 0 说明每一组都是全部成功或全部失败 |
| `Training/Actor:actor/policy_loss` | 裁剪后的策略梯度损失 |
| `Training/Actor:actor/approx_kl`、`actor/clip_fraction` | 一次更新让策略变化了多少；数值小说明更新稳定 |
| `Training/Actor:actor/grad_norm` | 裁剪前的梯度范数 |
| `Training/Actor:actor/lr` | 学习率 |

**时间（每步的秒数）**

| 指标 | 含义 |
| --- | --- |
| `Time:step` | 整个训练步 |
| `Time:generate_rollouts` | rollout 阶段：场景重置（`env/env/bootstrap_step`）、仿真和策略推理（`rollout/predict`） |
| `Time:actor_training` | 策略更新 |
| `Time:sync_weights` | 把新权重从 actor 发给 rollout worker |
| `Time:eval` | 评估（只在评估步出现） |

SFT 策略本身很强，成功率一开始就在 0.9 左右，所以这 40 个小步主要是为了看到完整的 RL 流程跑起来，而不是追求大的提升。注意每步只有 16 个回合时，`Environment:success_once` 的波动有多大。

## 5. 评估结果和视频

在第 10、20、30、40 步之后，RLinf 会在 32 个未参与训练的场景上评估策略，并为每个 GPU worker、每次评估写一个视频到 `$RUN_DIR/video/eval/seed_<worker 种子>/<n>.mp4`，其中 `<n>` = 0、1、2、3 分别对应第一次、第二次……评估。每个视频把该 worker 的 8 个评估环境拼成宫格，画面上叠加了每个回合的奖励和终止标志。每个 50 个动作的动作块录一帧，所以一个 400 个动作的回合约 10 帧；最后一帧显示的已经是为下一个回合重置后的场景。

In [ ]:
from IPython.display import HTML
import base64

EVAL_INDEX = -1   # 显示哪一次评估：-1 = 最新一次，0 = 第 10 步后，1 = 第 20 步后，……

rows = parse_metrics(Path(RUN_DIR, "metrics.log"))
evals = [r for r in rows if "Evaluation:success_once" in r]
if not evals:
    print("还没有评估结果：第一次评估在第 10 步之后。")
else:
    print(f"{'第几步后':>8} {'成功率':>7} {'回合数':>9} {'success_at_end':>15}")
    for r in evals:
        n = int(r.get("Evaluation:num_trajectories", 0))
        print(f"{r['step']:>12} {r['Evaluation:success_once']:>10.3f} {round(r['Evaluation:success_once'] * n):>6}/{n:<4} "
              f"{r.get('Evaluation:success_at_end', 0):>15.3f}")
    idx = EVAL_INDEX % len(evals)
    videos = sorted(Path(RUN_DIR, "video", "eval").glob(f"seed_*/{idx}.mp4"))
    print(f"\n第 {evals[idx]['step']} 步后评估的视频：{len(videos)} 个文件")
    display(HTML("".join(
        f"<div style='display:inline-block;margin:4px;text-align:center'><video width='420' controls loop autoplay muted "
        f"src='data:video/mp4;base64,{base64.b64encode(v.read_bytes()).decode()}'></video><br>{v.parent.name}/{v.name}</div>"
        for v in videos)))

## 6. 停止训练并清理

训练会在第 40 步的评估之后自动结束（启动后约 55 分钟）。**如果 workshop 结束时训练还在运行，请运行这个 cell 释放 GPU。** 也可以用它提前停止训练。它会中断训练进程组，等待 Ray 退出，然后停止容器里残留的 Ray 进程。

In [ ]:
%%bash
PID_FILE="${RUN_DIR}/train.pid"
is_running() { local s; s=$(ps -o stat= -p "$1" 2>/dev/null); [[ -n "${s}" && "${s}" != Z* ]]; }
if [[ -f "${PID_FILE}" ]] && is_running "$(cat "${PID_FILE}")"; then
    PID=$(cat "${PID_FILE}")
    echo "正在停止训练（进程组 ${PID}）……"
    kill -INT -- "-${PID}" 2>/dev/null || true
    for i in $(seq 1 60); do is_running "${PID}" || break; sleep 1; done
    if is_running "${PID}"; then echo "仍在运行，发送 SIGTERM"; kill -TERM -- "-${PID}" 2>/dev/null || true; sleep 10; fi
    source "${VENV}/bin/activate" && ray stop >/dev/null 2>&1 || true
    echo "已停止。"
else
    echo "没有正在运行的训练进程。"
fi
first=${GPU_GROUP%-*}; last=${GPU_GROUP#*-}
echo "GPU 组 ${GPU_GROUP}："
rocm-smi --showuse --showmemuse --csv | awk -F, -v a="${first}" -v b="${last}" \
    '/^card/ {n = substr($1, 5) + 0; if (n >= a && n <= b) printf "  %s: GPU use %s%%  VRAM %s%%\n", $1, $2, $3}'